**Présentation du problème et données**

https://www.kaggle.com/competitions/titanic/overview

**Problématique**
**Objectif de prédiction**
**variables (colonnes)**

**Défis associés**


In [401]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


**Préanalyse du dataset :**

In [402]:


df = pd.read_csv("train.csv")
print("shape ====================================")
print(df.shape)
print("head ====================================")
print(df.head())
print("info ====================================")
print(df.info())
print("describe ====================================")
print(df.describe())
print("Passagers survivants:", df["Survived"].sum())
print("Passagers décédés:", 891-df["Survived"].sum())
print("Ticket class", df["Pclass"].value_counts().sort_index())
print("null values ====================================")
print(df.isnull().sum())


print("Valeurs manquantes TRAIN :")
print(train.isnull().sum())

print("Valeurs manquantes TEST :")
print(test.isnull().sum())


shape ====================================
(891, 12)
head ====================================
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0     

**Résultats :**
**************** REFAIRE AVEC TEST ET TRAIN CAR IL FAUT CLEAN LES DEUX ****************************

On peut voir, grâce à `df.isnull().sum()`, l'ensemble des données manquantes que l'on devra traiter.

La colonne `Age` contient 177 valeurs manquantes. Plusieurs options sont disponibles pour régler ce problème. Nous pourrions supprimer la colonne, mais l'âge pourrait être un facteur très important dans la survie d'une personne. On pourrait également supprimer les lignes contenant des données manquantes ou modifier les données en attribuant à chaque case manquante l'âge médian des passagers.

Dans le cas de `Cabin`, la majorité des données sont manquantes. Le mieux serait donc de supprimer la colonne. Supprimer les lignes ayant une cabine manquante ferait en sorte que l'on perdrait la majorité de nos données.

`Cabin` pourrait jouer un rôle dans la survie de ses occupants à cause de la manière dont le bateau a coulé et de l'emplacement de la cabine. Par contre, cela ne veut pas dire que les occupants se trouvaient dans leur cabine lors de l'incident. Étant donné le nombre important de valeurs manquantes et cette incertitude, supprimer la colonne serait un choix raisonnable.

`Embarked`: Ports ou les passagers embarquent, il manque 2 valeurs. Le nombre de valeurs manquantes est assez basse pour les supprimer, ou on pourrait lui donner la valeur la plus fréquente. Dans ce cas nous allons enlever ces lignes manquantes vu que cela ne représente que 2/891

**Statistiques :** `df.isnull().sum()` va donner les statistiques du jeu de données:

`PassengerId` ID du passager. Il y a en tout 891 passagers à bord.

`Survived`: 342 personnes ont survécu, alors que 549 sont morts

`PCclass`: 216 Personnes ont un ticket de classe 1, 184 de classe 2 et 491 de classe 3.

**Colonnes à Standardiser**:
`Sex`: il faut transformer les colonnes en code binaire. Dans ce cas homme de 0 et femme 1 pour rendre la lecture de la colonne lisible par l'algorithme de ML


**Nettoyage des données**

In [403]:
# garder pour validation kaggle
passenger_ids = test["PassengerId"].copy()

# Supprimer le ID
train = train.drop(columns=["PassengerId"])
test = test.drop(columns=["PassengerId"])


# Supprimer name
train = train.drop(columns=["Name"])
test = test.drop(columns=["Name"])

# Supprimer ticket
# **************** test avec ticket supprimé, il est aussi possible de la garder semble plus compliqué à voir
train = train.drop(columns=["Ticket"])
test = test.drop(columns=["Ticket"])

#  Supprimer cabin
train = train.drop(columns=["Cabin"])
test = test.drop(columns=["Cabin"])
# Embarked
train = train.dropna(subset=["Embarked"])

# encoding de embarked
train = pd.get_dummies(train, columns=["Embarked"], dtype=int)
test = pd.get_dummies(test, columns=["Embarked"], dtype=int)

# age
median_age = train["Age"].median()
train["Age"] = train["Age"].fillna(median_age)
test["Age"] = test["Age"].fillna(median_age)

# supprimer age empire le résultat
# test = test.drop(columns=["Age"])
# train = train.drop(columns=["Age"])

# Sex

train["Sex"] = train["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})

# Fare 
# Il manque un seul fare dans test et on remplace par la médiane pour par briser le test kaggle
# car il manquerait une ligne sinon
median_fare = train["Fare"].median()
test["Fare"] = test["Fare"].fillna(median_fare)

# tests pour améliorer modele
# création colonne FamilySize
# création colonne isAlone
for data in [train, test]:
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)



# # test résultat standardisation
# print("Train :", train.shape)
# print("Test :", test.shape)

# print("Valeurs manquantes TRAIN : \n")
# print(train.isnull().sum())

# print("Valeurs manquantes TEST : \n")
# print(test.isnull().sum())

# print("Types des variables : \n")
# print(train.dtypes)

# Y représente la variable à prédire
y = train["Survived"] 
# Supprimer le Y
train = train.drop(columns=["Survived"])


In [404]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)

params_lr = {
        "logisticregression__C": [
        0.0001, 0.001, 0.005,
        0.01, 0.05, 0.1, 0.2,
        0.5, 1, 2, 5, 10,
        20, 50, 100, 500, 1000
    ],
    "logisticregression__class_weight": [None, "balanced"]
}

grid_lr = GridSearchCV(
    lr,
    params_lr,
    cv=5,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1"
    },
    refit="accuracy",
    n_jobs=-1
)

grid_lr.fit(train, y)

lr_predictions = grid_lr.predict(test)

print("Meilleurs paramètres :", grid_lr.best_params_)
print("Accuracy moyenne CV :", grid_lr.best_score_)

print(
    "F1-score moyen CV :",
    grid_lr.cv_results_["mean_test_f1"][grid_lr.best_index_]
)

Meilleurs paramètres : {'logisticregression__C': 0.05, 'logisticregression__class_weight': None}
Accuracy moyenne CV : 0.794172538564083
F1-score moyen CV : 0.7168624145000289


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
# max_iter augmente la précision
nn = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        solver="lbfgs",
        max_iter=100000,
        random_state=42
    )
)
# test de plusieurs (bn neurones couche 1, nb couche2)
# (8,2 donne meilleur res)
params = {
    "mlpclassifier__hidden_layer_sizes": [
        # (4,),
        # (8,),
        # (16,),
        # (32,),
        # (64,),
        # (128,),
        (8, 2),
        # (8, 4),
        # (16, 4),
        # (16, 8),
        # (32, 8),
        # (32, 16),
        # (64, 16),
        # (64, 32),
        # (16, 8, 4),
        # (32, 16, 8),
        # (64, 32, 16),
        # (128, 64, 32),
        # (32, 16, 8, 4),
        # (64, 32, 16, 8),
    ],
    "mlpclassifier__alpha": [0.01, 0.1, 1.0],
    "mlpclassifier__activation": ["relu", "tanh"]
}

grid_nn = GridSearchCV(
    nn,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_nn.fit(train, y)


print("Meilleurs paramètres :", grid_nn.best_params_)
print("Accuracy moyenne CV :", grid_nn.best_score_)

nn_predictions = grid_nn.predict(test)


Meilleurs paramètres : {'mlpclassifier__activation': 'relu', 'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (8, 2)}
Accuracy moyenne CV : 0.815552593156859


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd

# Modèles à comparer
models = {
    "Logistic Regression": grid_lr.best_estimator_,
    "Neural Network": grid_nn.best_estimator_
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        train,
        y,
        cv=cv,
        scoring=["accuracy", "f1"],
        n_jobs=-1
    )

    results.append({
        "Modèle": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "F1-score": scores["test_f1"].mean()
    })

pd.DataFrame(results).sort_values(
    "Accuracy",
    ascending=False
)

,Modèle,Accuracy,F1-score
1,Neural Network,0.813299,0.738748
0,Logistic Regression,0.798661,0.722830


In [407]:
import pandas as pd

# Régression logistique
submission_lr = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": lr_predictions
})

submission_lr.to_csv("./output/submission_lr.csv", index=False)

# Réseau de neurones
submission_nn = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": nn_predictions
})

submission_nn.to_csv("./output/submission_nn.csv", index=False)

# Vérification
print("Logistic Regression :", submission_lr.shape)
print("Neural Network :", submission_nn.shape)

Logistic Regression : (418, 2)
Neural Network : (418, 2)
